# Modeling — Baseline to Neural Network

Three models, in order of complexity:
1. **DummyRegressor** — predicts the mean ($9,589) every time. The true floor.
2. **LinearRegression** — interpretable benchmark. Tells me which features the linear model actually uses.
3. **Neural Network** — main model, exported to TFLite for the Flutter mobile app.

I always run the baseline before anything else. There's no point building something complex without knowing what lazy looks like.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')
with open('../data/processed/scaler_y.pkl','rb') as f: scaler_y = pickle.load(f)

feature_cols = [c for c in train_df.columns if c not in ['loan_amnt_scaled','loan_amnt_raw']]

X_train    = train_df[feature_cols].values
y_train_sc = train_df['loan_amnt_scaled'].values

X_test     = test_df[feature_cols].values
y_test_sc  = test_df['loan_amnt_scaled'].values
y_test_raw = test_df['loan_amnt_raw'].values

n_features = X_train.shape[1]
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'TensorFlow: {tf.__version__}')

## Model 1 — Dummy Baseline

Predicts the training mean ($9,589) for every single input. No features used at all.

> **Expected RMSE: ~$6,347** — this is the standard deviation of loan amounts.
> **R²: 0.0** — by definition (predicting the mean always gives R²=0).

Anything that doesn't beat this is not a model, it's noise.

In [ ]:
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train_sc)
y_pred_dm_sc = dummy.predict(X_test)
y_pred_dm = scaler_y.inverse_transform(y_pred_dm_sc.reshape(-1,1)).flatten()

rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_dm))
r2   = r2_score(y_test_raw, y_pred_dm)
print(f'Dummy Regressor:')
print(f'  RMSE: ${rmse:,.0f}')
print(f'  R²:   {r2:.4f}')
print(f'  It predicts ${scaler_y.inverse_transform([[dummy.constant_[0]]])[0][0]:,.0f} for every applicant')
with open('../outputs/models/dummy.pkl','wb') as f: pickle.dump(dummy, f)

## Model 2 — Linear Regression

Why linear regression before the neural network?
- It's a hard benchmark to beat on tabular data with weak features
- The coefficients tell me which features the model relies on — if they make intuitive sense, I trust the data
- If the neural network barely improves over linear regression, the extra complexity isn't worth the deployment overhead

I train it on scaled X with unscaled y — linear regression handles both fine without needing y scaled.

In [ ]:
# Train linear regression on scaled X, raw y
from sklearn.model_selection import train_test_split
import json
with open('../data/processed/feature_names.pkl','rb') as f: feature_names = pickle.load(f)
with open('../data/processed/scaler_X.pkl','rb') as f: scaler_X = pickle.load(f)

df_raw = pd.read_csv('../data/raw/credit_risk.csv')
df_raw['person_emp_length'] = df_raw['person_emp_length'].fillna(df_raw['person_emp_length'].median())
df_raw['loan_int_rate']     = df_raw['loan_int_rate'].fillna(df_raw.groupby('loan_grade')['loan_int_rate'].transform('median'))
cat_cols = ['person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file']
df_enc = pd.get_dummies(df_raw, columns=cat_cols, drop_first=True)
y_all = df_enc['loan_amnt']
X_all = df_enc.drop(columns=['loan_amnt','loan_percent_income','loan_status'])
_, _, y_train_raw, y_test_raw2 = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train_raw)
y_pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test_raw2, y_pred_lr))
mae_lr  = mean_absolute_error(y_test_raw2, y_pred_lr)
r2_lr   = r2_score(y_test_raw2, y_pred_lr)
print(f'Linear Regression:')
print(f'  RMSE: ${rmse_lr:,.0f}')
print(f'  MAE:  ${mae_lr:,.0f}')
print(f'  R²:   {r2_lr:.4f}')
with open('../outputs/models/linear_regression.pkl','wb') as f: pickle.dump(lr, f)

In [ ]:
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': lr.coef_})
coef_df['abs_coef'] = np.abs(coef_df['Coefficient'])
coef_df = coef_df.sort_values('abs_coef', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#C62828' if c > 0 else '#1565C0' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient ($ change in predicted loan amount per 1-std-dev increase)')
ax.set_title('Income and interest rate drive the linear model — but magnitudes are modest', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## Model 3 — Neural Network (Regression, TFLite-Ready)

### Key differences from a classification network

- **Output layer**: `Dense(1, activation='linear')` — not sigmoid. The output is an unbounded continuous value (scaled loan amount), not a probability.
- **Loss**: `mse` (mean squared error) — not binary_crossentropy.
- **Metric**: `mae` — more interpretable than raw MSE for regression.
- **Target**: scaled y (StandardScaler), inverse-transformed after prediction.

### Architecture
Same pyramid structure as before (128→64→32→16→1) with BatchNorm and Dropout. The first layer has 128 units — slightly wide for 20 features, but gives the model room to learn interaction effects between income, grade, and intent.

In [ ]:
def build_model(n_features):
    model = Sequential([
        Dense(128, input_shape=(n_features,), activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='linear'),  # ← linear output for regression
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

nn = build_model(n_features)
nn.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=6, monitor='val_loss', restore_best_weights=True),
    ModelCheckpoint('../outputs/models/best_nn.keras', monitor='val_loss', save_best_only=True)
]

history = nn.fit(
    X_train, y_train_sc,
    epochs=60, batch_size=64, validation_split=0.15,
    callbacks=callbacks, verbose=1
)
print(f'\nStopped at epoch {len(history.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history.history['loss'])+1)

axes[0].plot(ep, history.history['loss'],     label='Train', lw=2)
axes[0].plot(ep, history.history['val_loss'], label='Val',   lw=2, linestyle='--')
axes[0].set_title('MSE loss — scaled target means loss values are near 0–1, not millions', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (on scaled y)'); axes[0].legend()

axes[1].plot(ep, history.history['mae'],     label='Train MAE', lw=2)
axes[1].plot(ep, history.history['val_mae'], label='Val MAE',   lw=2, linestyle='--')
axes[1].set_title('MAE (scaled) — convert to dollars: multiply by scaler_y.scale_[0]', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (scaled)'); axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## TFLite Conversion — For the Flutter App

The Flutter mobile app loads a `.tflite` model. Converting from Keras is straightforward for a dense-only network.

**Important for inference in the app**: inputs must be scaled with `scaler_X` before feeding to the model, and the output must be inverse-transformed with `scaler_y` to get dollar amounts. The scaler parameters are:
- `scaler_y.mean_[0]` — loan amount mean (~$9,589)
- `scaler_y.scale_[0]` — loan amount std (~$6,347)
- Raw dollar prediction = `(model_output × scale) + mean`

In [ ]:
best_nn = tf.keras.models.load_model('../outputs/models/best_nn.keras')

# Sanity check: run one prediction
test_input = X_test[:1]
pred_scaled = best_nn.predict(test_input, verbose=0).flatten()[0]
pred_dollars = scaler_y.inverse_transform([[pred_scaled]])[0][0]
print(f'Sample prediction: scaled={pred_scaled:.4f} → ${pred_dollars:,.0f}')
print(f'Actual:                              ${y_test_raw[0]:,.0f}')

print(f'\nscaler_y.mean_[0]  = {scaler_y.mean_[0]:,.2f}  (training mean loan amount)')
print(f'scaler_y.scale_[0] = {scaler_y.scale_[0]:,.2f}  (training std loan amount)')
print('Flutter must apply: prediction_dollars = (raw_output × scale) + mean')
print(f'\nValid input range for loan amount demos: $500 – $35,000')

## Experiment Log

| # | Experiment | Change | RMSE Before | RMSE After | Decision |
|---|-----------|--------|-------------|------------|----------|
| 01 | Dummy baseline | Mean prediction | — | $6,347 | Benchmark set |
| 02 | Original code (buggy) | No scaling, dropped 2 cols, binary classification | N/A | N/A | Wrong task entirely |
| 03 | Fix to regression | Change target to loan_amnt, linear output | Dummy $6,347 | $5,916 (LR) | Correct problem defined |
| 04 | Drop loan_percent_income | Remove leaky feature | $5,916 | Cleaner model | Kept |
| 05 | Scale y with StandardScaler | MSE loss stabilised | Training diverged | Converges smoothly | Kept |
| 06 | Add BatchNormalization | After first two Dense layers | val_loss unstable | Stable training | Kept |
| 07 | Add Dropout (0.3, 0.2) | After BatchNorm | val_loss higher | val_loss lower | Kept |
| 08 | Final NN | All fixes | LR R²=0.131 | **NN R²=0.230** | Deployed |